# Final Usage E1 refit

Train **one fresh model on all 32,772 eligible teacher development rows**.
Keep E1 fixed: 30 epochs, seed 2753. Save the last epoch.

Use a fresh **Colab GPU**. First publish the local code to GitHub.
Reuse the existing Drive data ZIP. No new ZIP is needed.
The old five-model scores do not belong to this new model.

## 1. Mount Drive

In [1]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import drive

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "task-3-gender-usage-classification"
REPO_DIR = Path("/content/MLA2")
DRIVE_PROJECT = Path("/content/drive/MyDrive/MLA2")
DATA_ZIP = DRIVE_PROJECT / "data/task3-data.zip"
DRIVE_TASK_DIR = DRIVE_PROJECT / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"
drive.mount("/content/drive", force_remount=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Fetch code

This fetches the current branch from GitHub. Push any later code changes before starting.
Use a fresh runtime so imported code matches the checkout.

In [2]:
def run_checked(command):
    return subprocess.run([str(x) for x in command], check=True)


if (REPO_DIR / ".git").is_dir():
    remote = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True
    ).strip()
    if remote != REPO_URL:
        raise RuntimeError("The local checkout belongs to another repository")
    run_checked(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH])
    run_checked(["git", "-C", REPO_DIR, "switch", BRANCH])
    run_checked(["git", "-C", REPO_DIR, "merge", "--ff-only", f"origin/{BRANCH}"])
else:
    run_checked(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
print("Code commit:")
run_checked(["git", "rev-parse", "HEAD"])

Code commit:


CompletedProcess(args=['git', 'rev-parse', 'HEAD'], returncode=0)

## 3. Reuse teacher images

Read only eligible development paths from the canonical split.
Extract missing images from the existing ZIP. Keep the split and class map from GitHub.

In [3]:
import pandas as pd

from fashion.data import get_samples, load_splits

splits = load_splits(REPO_DIR / "data/processed/splits.csv")
paths = get_samples(splits, partition="development", target="usage").path.tolist()
missing = [source_path for source_path in paths if not (REPO_DIR / source_path).is_file()]
if missing:
    local_zip = Path("/content/task3-usage-e1-refit-data.zip")
    shutil.copyfile(DATA_ZIP, local_zip)
    with zipfile.ZipFile(local_zip) as archive:
        for source_path in missing:
            if not source_path.startswith("data/raw/teacher/train/images_train/"):
                raise ValueError(f"Unexpected development image path: {source_path}")
            target = (REPO_DIR / source_path).resolve()
            if not target.is_relative_to(REPO_DIR.resolve()):
                raise ValueError("Image path leaves the repository")
            target.parent.mkdir(parents=True, exist_ok=True)
            partial = target.with_suffix(target.suffix + ".partial")
            with archive.open(source_path) as source, partial.open("wb") as output:
                shutil.copyfileobj(source, output)
            partial.replace(target)
print(f"Development images ready: {len(paths):,}")

Development images ready: 32,772


## 4. Check the frozen recipe and inputs

SmallCNN: **391,209 parameters**, channels 32/64/128/256, global average pooling.
RGB input: height 80, width 60. No dropout or image augmentation.
Plain cross-entropy. AdamW: rate 0.001, weight decay 0.0001, batch 128.
Cosine schedule: 30 epochs, ending at 0.00001.

Normalization uses all admitted development content pixels. Padding is left out.
Holdout, quarantine, test and external images stay out of training.

In [4]:
import torch
from fashion.train.task3_usage_e1_refit import EXPERIMENT, prepare_refit

config, contract, training = prepare_refit(root=REPO_DIR)
print("Rows:", len(training), "Classes:", contract["class_names"])
print("Recipe:", config.to_dict())
print("PyTorch:", torch.__version__)
print("Output:", DRIVE_TASK_DIR / "experiments" / EXPERIMENT / "usage")

Checking 32,772 development image hashes
Rows: 32772 Classes: ['Casual', 'Ethnic', 'Formal', 'Home', 'NA', 'Party', 'Smart Casual', 'Sports', 'Travel']
Recipe: {'target': 'usage', 'image_height': 80, 'image_width': 60, 'channels': [32, 64, 128, 256], 'batch_size': 128, 'epochs': 30, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'minimum_learning_rate': 1e-05, 'seed': 2753, 'num_workers': 2, 'mixed_precision': False, 'early_stopping': False, 'augmentation': 'none', 'loss_name': 'cross_entropy', 'optimizer_name': 'AdamW', 'scheduler_name': 'CosineAnnealingLR', 'checkpoint_rule': 'final_epoch', 'model_family': 'task3_small_cnn', 'scratch': True, 'submission_eligible': True, 'num_classes': 9}
PyTorch: 2.11.0+cu128
Output: /content/drive/MyDrive/MLA2/task3/experiments/t3_usage_e1_teacher_all_development_refit/usage


## 5. Train once and save

This cell starts the real fit. It uses no validation set or early stopping.
A repeat run verifies and reuses completed files. A failed or interrupted run stops
for inspection. It never silently resumes or replaces a saved fit.

The Drive registry is the main log. The local log is a mirror. Other task rows stay safe.

In [5]:
from fashion.train.task3_usage_e1_refit import run_usage_e1_refit

result = run_usage_e1_refit(
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=(LOCAL_REGISTRY,),
)
print("Status:", result["status"], "Reused:", result["reused"])
print("Manifest:", result["manifest_path"])

Checking 32,772 development image hashes
Epoch 1/30: training loss 0.5759
Epoch 2/30: training loss 0.4139
Epoch 3/30: training loss 0.3822
Epoch 4/30: training loss 0.3401
Epoch 5/30: training loss 0.3237
Epoch 6/30: training loss 0.3063
Epoch 7/30: training loss 0.2868
Epoch 8/30: training loss 0.2657
Epoch 9/30: training loss 0.2473
Epoch 10/30: training loss 0.2280
Epoch 11/30: training loss 0.2087
Epoch 12/30: training loss 0.1972
Epoch 13/30: training loss 0.1702
Epoch 14/30: training loss 0.1569
Epoch 15/30: training loss 0.1406
Epoch 16/30: training loss 0.1200
Epoch 17/30: training loss 0.1007
Epoch 18/30: training loss 0.0952
Epoch 19/30: training loss 0.0755
Epoch 20/30: training loss 0.0829
Epoch 21/30: training loss 0.0516
Epoch 22/30: training loss 0.0436
Epoch 23/30: training loss 0.0387
Epoch 24/30: training loss 0.0341
Epoch 25/30: training loss 0.0299
Epoch 26/30: training loss 0.0264
Epoch 27/30: training loss 0.0251
Epoch 28/30: training loss 0.0235
Epoch 29/30: tra

## 6. Check saved training history

In [6]:
manifest_dir = Path(result["manifest_path"]).parent
history = pd.read_csv(manifest_dir / "history.csv")
assert history.epoch.tolist() == list(range(1, 31))
assert history.training_rows.eq(32772).all()
assert history.selected_checkpoint.tolist() == [False] * 29 + [True]
assert result["training_completed"] and not result["evaluation_completed"]
display(history.tail())
print("Checkpoint:", manifest_dir / "final_epoch.pt")
print("Normalization:", manifest_dir / "normalization.json")

,epoch,learning_rate,train_loss,training_rows,selected_checkpoint
25,26,0.000076,0.026440,32772,False
26,27,0.000053,0.025060,32772,False
27,28,0.000034,0.023469,32772,False
28,29,0.000021,0.022246,32772,False
29,30,0.000013,0.021569,32772,True


Checkpoint: /content/drive/MyDrive/MLA2/task3/experiments/t3_usage_e1_teacher_all_development_refit/usage/final_epoch.pt
Normalization: /content/drive/MyDrive/MLA2/task3/experiments/t3_usage_e1_teacher_all_development_refit/usage/normalization.json


## 7. Hand off the fixed model for judgement

Use `model_manifest.json` to check file hashes. Load only this `final_epoch.pt`
and its saved full-development normalization. Use one model in evaluation mode,
then softmax and argmax in the saved nine-class order. Do not average old folds.

Evaluate later in a new evidence folder. Report accuracy, nine-class macro-F1,
per-class failures and inference cost. Keep all scores separate from old E1 scores.
Do not tune the recipe or pick another epoch from those results.

Holdout and recovered test labels were already viewed. They are **not newly blind**.
This notebook does not evaluate the model or write submission predictions.
The fixed five-column teacher submission format still applies.